# Audio to Text - faster-whisper + Free GPU

> Model: faster-whisper large-v3 / turbo
> Platform: Google Colab (T4) or AWS Studio Lab

---


## Step 0 - Check GPU

Colab: Runtime -> Change runtime type -> **GPU (T4)**
Studio Lab: Start with **GPU** option


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU available:', r.stdout.strip())
else:
    print('No GPU detected. Please switch runtime to GPU and re-run.')


## Step 1 - Install


In [ ]:
# faster-whisper does NOT require system ffmpeg
# Uncomment next line if your audio is .m4a / .aac
# !apt-get install -y -q ffmpeg
!pip install -q faster-whisper


## Step 2 - Mount Google Drive

Put your audio file in Google Drive root, then mount here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
print('Drive root contents:')
for f in sorted(os.listdir(DRIVE_ROOT))[:20]:
    print(' ', f)


## Step 3 - Config

| Param | Default | Notes |
|-------|---------|-------|
| MODEL_SIZE | large-v3 | Best accuracy; use turbo for speed |
| LANGUAGE | zh | Chinese; None = auto-detect |
| BATCH_SIZE | 16 | T4 has ~15GB VRAM, 16 is stable |
| VAD_FILTER | True | Skip silence to save time |
| WORD_TIMESTAMPS | True | Word-level timestamps for subtitles |


In [ ]:
# ====== Edit here ======
AUDIO_PATH  = '/content/drive/MyDrive/audio.mp3'  # your audio file
OUTPUT_PATH = '/content/drive/MyDrive/transcript.txt'
SRT_PATH    = '/content/drive/MyDrive/transcript.srt'

MODEL_SIZE      = 'large-v3'  # tiny/base/small/medium/turbo/large-v3
LANGUAGE        = 'zh'         # zh/en/ja or None for auto
BATCH_SIZE      = 16
VAD_FILTER      = True
WORD_TIMESTAMPS = True
# =======================

import os
assert os.path.exists(AUDIO_PATH), f'File not found: {AUDIO_PATH}'
print(f'Audio: {os.path.getsize(AUDIO_PATH)/1024/1024:.1f} MB')


## Step 4 - Load Model and Transcribe


In [ ]:
import time
from faster_whisper import WhisperModel, BatchedInferencePipeline

print(f'Loading model {MODEL_SIZE} ...')
t0 = time.time()
model = WhisperModel(MODEL_SIZE, device='cuda', compute_type='float16')
batched = BatchedInferencePipeline(model=model)
print(f'Model loaded in {time.time()-t0:.1f}s')

print('Transcribing...')
t1 = time.time()
segments, info = batched.transcribe(
    AUDIO_PATH,
    batch_size=BATCH_SIZE,
    vad_filter=VAD_FILTER,
    word_timestamps=WORD_TIMESTAMPS,
    language=LANGUAGE,
)
segments = list(segments)
elapsed = time.time() - t1

print(f'Done! {len(segments)} segments in {elapsed:.1f}s')
print(f'Language: {info.language} ({info.language_probability:.2%})')
print(f'Duration: {info.duration:.1f}s  realtime: {info.duration/elapsed:.1f}x')


## Step 5 - Save Output


In [ ]:
# Plain text
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for seg in segments:
        f.write(seg.text.strip() + '\n')
print(f'Text saved: {OUTPUT_PATH}')

# SRT subtitle
def fmt(sec):
    h, r = divmod(int(sec), 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d},{int((sec % 1)*1000):03d}'

with open(SRT_PATH, 'w', encoding='utf-8') as f:
    for i, seg in enumerate(segments, 1):
        f.write(f'{i}\n{fmt(seg.start)} --> {fmt(seg.end)}\n{seg.text.strip()}\n\n')
print(f'SRT saved: {SRT_PATH}')

print('\n--- Preview (first 5) ---')
for seg in segments[:5]:
    print(f'[{seg.start:.1f}s->{seg.end:.1f}s] {seg.text.strip()}')


## Step 6 (Optional) - WhisperX Speaker Diarization

Use this to identify who is speaking.
Need: HF token from https://huggingface.co/settings/tokens
and accept license at https://huggingface.co/pyannote/speaker-diarization-3.1


In [ ]:
# Uncomment to enable speaker diarization (WhisperX)

# !pip install -q whisperx
# HF_TOKEN = 'hf_your_token_here'

# import whisperx
# audio = whisperx.load_audio(AUDIO_PATH)
# wx = whisperx.load_model('large-v3', 'cuda', compute_type='float16')
# result = wx.transcribe(audio, batch_size=16, language='zh')
# align_model, meta = whisperx.load_align_model(language_code='zh', device='cuda')
# result = whisperx.align(result['segments'], align_model, meta, audio, 'cuda')
# diarize = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device='cuda')
# result = whisperx.assign_word_speakers(diarize(audio), result)
# for seg in result['segments']:
#     print(f"[{seg.get('speaker','?')}] {seg['text'].strip()}")


---
## Note: Switch to AWS Studio Lab

Studio Lab keeps packages persistent. Only change the path:

```python
AUDIO_PATH = '/home/studio-lab-user/audio.mp3'
```

Everything else is identical.
